# 3. 差分の差法

この notebook では、**差分の差法（DiD）**実行します。その際に、推定対象を表す線型汎函数 $m$ を書くだけで二重機械学習を実行できることを確認します。

ここでの見方は次の通りです。

- まず、二期間データから差分アウトカム $\Delta Y = Y_1 - Y_0$ を作る
- DiD の ATT を $\theta = \mathbb{E}[m(X, \gamma_0)]$ の形に書く
- その $m$ を Python 関数として定義する
- あとは 二重機械学習とリース回帰 に渡して推定する

In [5]:
import numpy as np
import pandas as pd

from genriesz import (
    CallableFunctional,
    grr_functional,
    SquaredGenerator,
    PolynomialBasis,
    TreatmentInteractionBasis,
)

rng = np.random.default_rng(0)


## DGP

人工データは、**共変量 $Z$ に依存して処置が割り当てられ、かつ共通トレンドも $Z$ に依存する**ように作っています。

この設定の意図は明確です。  
もし共通トレンドが一様であれば、単純な DID 平均差でもかなりうまくいきます。そこで今回は、あえて

- 処置の受けやすさが $Z$ に依存する
- 時間トレンドも $Z$ に依存する

ようにして、**無条件の parallel trends が破れる状況**を作っています。

したがって、共変量を無視した naive DID はバイアスを持ちます。  
一方で、$Z$ で条件付けた回帰関数を使えば、条件付き平行トレンドのもとで ATT を回復できる、という構図です。

このセルの最後では、大きな母集団標本を用いて、naive DID が真値からずれることを確認します。


In [6]:
def draw_panel(n: int, tau: float, seed: int = 0):
    rng = np.random.default_rng(seed)
    Z = rng.normal(size=(n, 2))

    # treatment assignment depends on Z
    logits = 0.7 * Z[:, 0] - 0.5 * Z[:, 1]
    e = 1.0 / (1.0 + np.exp(-logits))
    D = rng.binomial(1, e, size=n).astype(int)

    # baseline and common trend both depend on Z
    mu = 0.6 * Z[:, 0] - 0.3 * Z[:, 1] ** 2
    trend = 0.5 + 0.8 * Z[:, 0] + 0.4 * Z[:, 1] ** 2

    # unit fixed effect
    u = rng.normal(scale=1.0, size=n)

    Y0 = mu + u + rng.normal(scale=1.0, size=n)
    Y1 = mu + trend + tau * D + u + rng.normal(scale=1.0, size=n)

    X = np.column_stack([D.astype(float), Z])  # X = [D, Z]
    return X, Y0, Y1, D

tau_true = 1.0

# Large population to show the naive DID bias
X_pop, Y0_pop, Y1_pop, D_pop = draw_panel(n=200_000, tau=tau_true, seed=1)
DeltaY_pop = Y1_pop - Y0_pop

naive_did_pop = DeltaY_pop[D_pop == 1].mean() - DeltaY_pop[D_pop == 0].mean()

print(f"True DID/ATT on ΔY (by construction): {tau_true:.4f}")
print(f"Naive DID without covariates        : {naive_did_pop:.4f}")


True DID/ATT on ΔY (by construction): 1.0000
Naive DID without covariates        : 1.4828


## 線型汎函数 $m$ を定義する

DiD では、差分アウトカム $\Delta Y = Y_1 - Y_0$ を考えると、関心のある量は「処置群における平均的な追加変化」です。  
ここで

$$
\gamma(d, z) = \mathbb{E}[\Delta Y \mid D=d, Z=z]
$$

を回帰関数とし、$\pi = \mathbb{P}(D=1)$ を処置群比率とすると、ATT は

$$
\theta_{\mathrm{ATT}} = \mathbb{E}\left[\frac{D}{\pi}\{\gamma(1,Z)-\gamma(0,Z)\}\right]
$$

と書けます。

この右辺は、回帰関数 $\gamma$ に線形に作用しています。したがって、

$$
m(x,\gamma) = \frac{d}{\pi}\{\gamma(1,z)-\gamma(0,z)\},
\quad x=(d,z)
$$

と置けば、

$$
\theta_{\mathrm{ATT}} = \mathbb{E}[m(X,\gamma_0)]
$$

という、スライドで扱った一般形そのものになります。

このセルでは、その $m$ を Python 関数 `make_did_functional` として定義しています。  

In [7]:
def make_did_functional(pi: float, treatment_index: int = 0):
    def m(x_row, gamma):
        x1 = np.array(x_row, dtype=float, copy=True)
        x0 = np.array(x_row, dtype=float, copy=True)
        x1[treatment_index] = 1.0
        x0[treatment_index] = 0.0

        d = float(x_row[treatment_index])
        return (d / pi) * (gamma(x1) - gamma(x0))

    return m


## 推定関数を作る

実装上のポイントは 3 つあります。

### 1. 回帰対象は $\Delta Y$

DiD を「一般形」に落とすため、まず $Y_1 - Y_0$ を作り、それをアウトカムとして扱います。  
すると、問題は「$\Delta Y$ の回帰関数に作用する線型汎函数を推定する」形になります。

### 2. 基底には `TreatmentInteractionBasis` を使う

今回は処置 $D$ が二値なので、処置ごとに異なる回帰関数を表現しやすい `TreatmentInteractionBasis` が自然です。  
共変量 $Z$ には二次多項式基底を当てています。

### 3. `cross_fit=True` で交差適合の有無を選択する

- `cross_fit=False` なら、同じデータで局外母数を学習し、そのまま使う
- `cross_fit=True` なら、fold ごとに外部サンプルで局外母数を学習し、推定対象の fold には out-of-fold 予測だけを使う

という違いになります。

In [8]:
# Sample used for estimation
X, Y0, Y1, D = draw_panel(n=1500, tau=tau_true, seed=0)
DeltaY = Y1 - Y0
pi_hat = float(D.mean())

# ATE / DID 向けには TreatmentInteractionBasis が自然
psi = PolynomialBasis(degree=2, include_bias=True)
phi = TreatmentInteractionBasis(base_basis=psi)

gen = SquaredGenerator(C=0.0).as_generator()

m_did = CallableFunctional(
    make_did_functional(pi=pi_hat, treatment_index=0),
    name="DID via custom m",
)

def fit_did_generic(*, cross_fit: bool, folds: int = 3):
    return grr_functional(
        X=X,
        Y=DeltaY,
        m=m_did,
        basis=phi,
        generator=gen,
        cross_fit=cross_fit,
        folds=folds,
        random_state=0,
        estimators=("ra", "rw", "arw", "tmle"),
        outcome_models="shared",
        outcome_link="identity",
        riesz_penalty="l2",
        riesz_lam=1e-3,
        max_iter=200,
        tol=1e-8,
    )


In [9]:
res_plugin = fit_did_generic(cross_fit=False)
res_dml = fit_did_generic(cross_fit=True, folds=3)

print("=== 交差適合なし ===")
print(res_plugin.summary_text())
print()
print("=== 交差適合あり ===")
print(res_dml.summary_text())


=== 交差適合なし ===
DID via custom m estimates (n=1500)
alpha=0.05 | null=0.0

Estimator         Estimate            SE                           CI     p-value
---------------------------------------------------------------------------------
RA                 1.00866     0.0268717        [ 0.955997,  1.06133]           0
RW                 1.00891       0.14102         [ 0.732513,  1.2853]    8.41e-13
ARW                1.00842     0.0912764        [ 0.829521,  1.18732]           0
TMLE               1.00842     0.0912753        [ 0.829523,  1.18732]           0

=== 交差適合あり ===
DID via custom m estimates (n=1500)
alpha=0.05 | null=0.0

Estimator         Estimate            SE                           CI     p-value
---------------------------------------------------------------------------------
RA                 1.01377      0.027939        [ 0.959008,  1.06853]           0
RW                 1.00934      0.141264        [ 0.732466,  1.28621]       9e-13
ARW                1.01159     

## 比較表の見方

この notebook では、`cross_fit=False` と `cross_fit=True` の両方を実行し、推定量を並べて比較しています。

表の見方は次の通りです。

- `estimate`: 推定値
- `se`: 推定標準誤差
- `ci_low`, `ci_high`: 信頼区間
- `error_vs_true`: 真値との差

In [10]:
def records_from_result(res, fit_label: str, true_value: float):
    rows = []
    for key, est in res.estimates.items():
        rows.append(
            {
                "fit": fit_label,
                "estimator": key.upper(),
                "estimate": est.estimate,
                "se": est.se,
                "ci_low": est.ci_low,
                "ci_high": est.ci_high,
                "error_vs_true": est.estimate - true_value,
            }
        )
    return rows

comparison_df = pd.DataFrame(
    records_from_result(res_plugin, "no cross-fit", tau_true)
    + records_from_result(res_dml, "cross-fit = DML", tau_true)
)

comparison_df = comparison_df.sort_values(["fit", "estimator"]).reset_index(drop=True)
comparison_df


,fit,estimator,estimate,se,ci_low,ci_high,error_vs_true
0,cross-fit = DML,ARW,1.011592,0.094698,0.825988,1.197196,0.011592
1,cross-fit = DML,RA,1.013768,0.027939,0.959008,1.068527,0.013768
2,cross-fit = DML,RW,1.009339,0.141264,0.732466,1.286212,0.009339
3,cross-fit = DML,TMLE,1.011637,0.094700,0.826028,1.197245,0.011637
4,no cross-fit,ARW,1.008420,0.091276,0.829521,1.187318,0.008420
5,no cross-fit,RA,1.008665,0.026872,0.955997,1.061333,0.008665
6,no cross-fit,RW,1.008907,0.141020,0.732513,1.285301,0.008907
7,no cross-fit,TMLE,1.008419,0.091275,0.829523,1.187316,0.008419


In [11]:
sample_naive_did = DeltaY[D == 1].mean() - DeltaY[D == 0].mean()

print(f"Sample naive DID without covariates: {sample_naive_did:.4f}")
print(f"Target (true tau)                  : {tau_true:.4f}")
print()
print("Cross-fitted ARW / TMLE are the generic DML-style estimators here.")


Sample naive DID without covariates: 1.4581
Target (true tau)                  : 1.0000

Cross-fitted ARW / TMLE are the generic DML-style estimators here.


## この notebook のまとめ

この notebook で確認したかったことは、まさに次の 3 点です。

- DiD の ATT も、一般形 $\theta = \mathbb{E}[m(X,\gamma_0)]$ で表せる。
- そのため、$m(x,\gamma)$ を書けば 二重機械学習 + リース回帰 = 自動バイアス除去学習で推定できる。

スライドで強調した「推定対象を線型汎函数として書き、その上で直交スコアと交差適合を使う」という考え方が、DiD にもそのまま適用できることが分かります。